# Predictor

Notebook de predição: lê os artefactos gerados pelo `data_ingestion.ipynb` e executa o pipeline de inferência completo.

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:
import numpy as np
import pandas as pd
import polars as pl

from api.config.dataset_config import DatasetConfig
from api.config.output_config import OutputConfig
from api.config.predictor_config import PredictorConfig
from api.config.training_config import TrainingConfig

from moviasai.data.dataset import GenerateDataInput
from moviasai.data.normalization import ProfileDatasetNormalizer
from moviasai.forecasting.predictor import ForecastingPredictor

In [3]:
# Configs
dataset_cfg = DatasetConfig.from_yaml('../config/dataset_config.yaml')
training_cfg = TrainingConfig.from_yaml('../config/training_config.yaml')
output_cfg = OutputConfig.from_yaml('../config/output_config.yaml')
predictor_cfg = PredictorConfig.from_yaml('../config/predictor_config.yaml')

print(f"num_weeks_recent: {dataset_cfg.num_weeks_recent}")
print(f"horizon_weeks:    {dataset_cfg.horizon_weeks}")
print(f"cv_max:           {training_cfg.normalization.cv_max}")
print(f"ratio_max:        {training_cfg.normalization.ratio_max}")

num_weeks_recent: 4
horizon_weeks:    4
cv_max:           5.0
ratio_max:        5.0


In [4]:
# Ler artefactos do data_ingestion
TMP = '../../tmp'

df_long = pd.read_csv(f'{TMP}/long.csv')
df_sampled = pl.read_csv(f'{TMP}/sample.csv', try_parse_dates=True)
metadata_km = pd.read_csv(f'{TMP}/metadata_km.csv')
metadata_h = pd.read_csv(f'{TMP}/metadata_h.csv')

print(f"df_long:     {df_long.shape}")
print(f"df_sampled:  {df_sampled.shape}")
print(f"metadata_km: {metadata_km.shape}")
print(f"metadata_h:  {metadata_h.shape}")

df_long:     (1319976, 4)
df_sampled:  (1368864, 4)
metadata_km: (10981, 6)
metadata_h:  (11304, 6)


In [ ]:
# Rótulos de qualidade por veículo
QUALITY_LABELS = {0: "Válida", 1: "Outlier", 2: "Não modelável", 3: "Vazia"}

for label, meta in [("KM", metadata_km), ("H", metadata_h)]:
    if "quality" in meta.columns:
        meta["quality_label"] = meta["quality"].map(QUALITY_LABELS).fillna("Desconhecida")
        print(f"\n[{label}] Distribuição de qualidade:")
        print(meta["quality_label"].value_counts().to_string())

In [5]:
def predict(target: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Executa o pipeline completo de predição para um target ('km' ou 'h').

    Returns
    -------
    (df_daily, df_heads)
        DataFrames com veiculo_id e predições denormalizadas.
    """
    meta = metadata_km if target == 'km' else metadata_h
    num_days = dataset_cfg.num_weeks_recent * 7

    # 1) Filtrar df_long pelo target
    df_target = df_long[df_long['feature_class'] == target].copy()

    # 2) Pivot: long → wide (veiculo_id, feat_1, ..., feat_n)
    df_profile = df_target.pivot(
        index='veiculo_id', columns='feature', values='valor',
    ).reset_index()
    df_profile.columns.name = None
    vehicle_ids = df_profile['veiculo_id'].values
    print(f"[{target}] df_profile: {df_profile.shape}")

    # 3) Upper dos veículos presentes em df_profile
    upper_map = meta.set_index('veiculo_id')['upper']
    upper = upper_map.reindex(vehicle_ids).values.astype(np.float64)

    # 4) df_recent: últimos num_days registos por veículo, alinhados seg-dom
    target_col = f'{target}_dia_clean'
    df_s = (
        df_sampled
        .filter(pl.col('veiculo_id').is_in(vehicle_ids.tolist()))
        .sort('veiculo_id', 'data')
    )

    # Truncar ao último domingo de cada veículo e pegar os últimos num_days
    df_s = df_s.with_columns(pl.col('data').dt.weekday().alias('_dow'))
    df_recent = (
        df_s
        .group_by('veiculo_id')
        .map_groups(lambda g: (
            g
            # Truncar ao último domingo (_dow == 7 em Polars)
            .filter(pl.col('data') <= g['data'].filter(g['_dow'] == 7).last())
            .tail(num_days)
        ))
        .drop('_dow')
        .sort('veiculo_id', 'data')
    )

    # Validar séries completas
    counts = df_recent.group_by('veiculo_id').len()
    valid_ids = counts.filter(pl.col('len') == num_days)['veiculo_id']
    n_dropped = len(vehicle_ids) - len(valid_ids)
    if n_dropped > 0:
        print(f"[{target}] {n_dropped} veículos descartados (série < {num_days} dias)")

    # Filtrar para veículos com série completa
    mask = np.isin(vehicle_ids, valid_ids.to_numpy())
    df_profile = df_profile[mask].reset_index(drop=True)
    upper = upper[mask]
    vehicle_ids = df_profile['veiculo_id'].values
    df_recent = df_recent.filter(pl.col('veiculo_id').is_in(vehicle_ids.tolist()))

    # Converter para pandas para GenerateDataInput
    df_recent_pd = df_recent.to_pandas()
    df_recent_pd['data'] = pd.to_datetime(df_recent_pd['data'])

    print(f"[{target}] veículos válidos: {len(vehicle_ids)}")
    print(f"[{target}] df_recent: {df_recent_pd.shape}")

    # 5) Gerar DataInput
    data_input = GenerateDataInput(
        target=target,
        df_profile=df_profile,
        df_recent=df_recent_pd,
        upper=upper,
    ).generate()
    print(f"[{target}] DataInput: X_general={data_input.X_general.shape}, X_recent={data_input.X_recent.shape}")

    # 6) Instanciar normalizer
    normalizer = ProfileDatasetNormalizer(
        cv_max=training_cfg.normalization.cv_max,
        ratio_max=training_cfg.normalization.ratio_max,
    )

    # 7) Normalizar
    data_input_norm = normalizer.fit_normalize_data_input(data_input)

    # 8) Instanciar predictor
    predictor = ForecastingPredictor.from_config(
        target=target,
        predictor_cfg=predictor_cfg,
        output_cfg=output_cfg,
        horizon_weeks=dataset_cfg.horizon_weeks,
    )

    # 9) Predição
    results = predictor.predict(data_input_norm)

    df_daily = pd.DataFrame(
        results['y_daily'],
        columns=[f'd_{i+1}' for i in range(results['y_daily'].shape[1])],
    )
    df_daily.insert(0, 'veiculo_id', vehicle_ids)

    df_heads = pd.DataFrame(
        results['y_heads'],
        columns=[f'head_{i+1}' for i in range(results['y_heads'].shape[1])],
    )
    df_heads.insert(0, 'veiculo_id', vehicle_ids)

    print(f"[{target}] df_daily: {df_daily.shape}, df_heads: {df_heads.shape}")
    return df_daily, df_heads

In [6]:
daily_km, heads_km = predict('km')

[km] df_profile: (8148, 79)
[km] veículos válidos: 8148
[km] df_recent: (228144, 4)
[km] DataInput: X_general=(8148, 78), X_recent=(8148, 28)
[km] df_daily: (8148, 8), df_heads: (8148, 5)


In [7]:
daily_h, heads_h = predict('h')

[h] df_profile: (8148, 78)
[h] veículos válidos: 8148
[h] df_recent: (228144, 4)
[h] DataInput: X_general=(8148, 77), X_recent=(8148, 28)
[h] df_daily: (8148, 8), df_heads: (8148, 5)


In [8]:
heads_km

,veiculo_id,head_1,head_2,head_3,head_4
0,4,134.766878,134.955942,135.386552,131.786095
1,11,211.881666,213.221560,214.161490,210.764164
2,38,0.716316,11.849469,26.079758,30.877873
3,81,2545.899160,2659.226486,2752.025685,2694.545638
4,105,2815.100018,2782.853991,2759.152344,2656.503637
...,...,...,...,...,...
8143,29415,581.020106,602.977227,598.286267,554.644852
8144,29417,45.886147,50.971715,62.147372,45.098399
8145,29419,23.463819,56.212221,82.016724,92.039617
8146,29426,78.943796,91.767940,95.770649,93.629698


In [9]:
daily_km

,veiculo_id,d_1,d_2,d_3,d_4,d_5,d_6,d_7
0,4,17.106017,18.717039,19.564241,18.489422,16.190405,0.876426,0.015780
1,11,23.157820,31.395495,33.718710,32.311652,29.146671,1.671836,0.025911
2,38,0.131988,2.317046,1.837354,0.000000,0.000000,0.000000,0.000000
3,81,114.476602,329.994203,397.226412,424.215114,387.139323,311.960245,223.338577
4,105,252.019294,404.942131,442.391030,445.058459,412.569753,332.659480,267.361820
...,...,...,...,...,...,...,...,...
8143,29415,69.494953,51.004064,31.638295,29.520878,36.563794,0.004932,0.033677
8144,29417,16.378516,21.824145,18.677253,20.777388,25.161690,2.940398,0.059382
8145,29419,0.000000,0.000000,0.000000,1.244178,2.996418,0.080954,0.023958
8146,29426,6.066328,8.375422,8.345884,7.464707,6.345531,0.803071,0.010811
